# VAIC2026 — Fine-tune Vietnamese Legal Reranker (PyTorch)

Fine-tunes `BAAI/bge-reranker-v2-m3` on Zalo Vietnamese legal retrieval + our in-domain
Điện Biên documents, then evaluates Recall@k / MRR / nDCG (retrieval → +reranker → +fine-tuned).

**Before running:** Settings → Accelerator → **GPU T4 x1** (or P100), and Internet → **On**.

Attach the `finetune/` folder as an input dataset named **vaic-finetune** (see KAGGLE.md).
Paths below assume `/kaggle/input/vaic-finetune/`.

In [ ]:
# Kaggle preinstalls torch/transformers/datasets. Add the two light extras.
!pip install -q rank-bm25 sentence-transformers

In [ ]:
import os, glob
# Locate the attached dataset (folder name may vary slightly).
cands = glob.glob('/kaggle/input/*/notebooks/train_reranker.py')
assert cands, 'Attach the finetune folder as an input dataset first (see KAGGLE.md).'
SRC = os.path.dirname(os.path.dirname(cands[0]))
DATA = os.path.join(SRC, 'data')
NB = os.path.join(SRC, 'notebooks')
OUT = '/kaggle/working/bge-reranker-dienbien'
print('SRC :', SRC)
print('DATA:', DATA, '->', os.listdir(DATA))

## 1. Train

Native PyTorch loop (AdamW + linear warmup + AMP + grad clipping). Zalo triples are
built with BM25 hard negatives, combined with our in-domain triples. ~15–30 min on T4.

Tip: add `--max-queries 300` for a fast first pass; drop it for the full run.

In [ ]:
!python {NB}/train_reranker.py \
    --base-model BAAI/bge-reranker-v2-m3 \
    --indomain {DATA}/train_indomain.jsonl \
    --out {OUT} \
    --epochs 2 --batch-size 16 --max-len 512 --neg-per-pos 7 --num-workers 2

## 2. Evaluate (retrieval vs +reranker base vs +fine-tuned)

Same candidate pools for all three rankers. This is the before/after table for the deck.

In [ ]:
!python {NB}/eval_reranker.py \
    --chunks {DATA}/chunks.jsonl \
    --eval {DATA}/eval_indomain.jsonl \
    --base-model BAAI/bge-reranker-v2-m3 \
    --finetuned {OUT}

## 3. Package the model for download / serving

Zip `/kaggle/working/bge-reranker-dienbien` and download it, or **Save Version** and
publish the output as a Kaggle model to load in the retrieval-api.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/bge-reranker-dienbien', 'zip', OUT)
print('zipped ->', '/kaggle/working/bge-reranker-dienbien.zip')